<a href="https://colab.research.google.com/github/Pri-codes-10/Kaggle_housing_competition/blob/main/kaggle_housing_advancedregression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import skew

train_df = pd.read_csv("train (1).csv")
test_df = pd.read_csv("test (1).csv")
test_ids = test_df["Id"]

OUTLIER REMOVAL


In [2]:
train_df = train_df[
    ~((train_df["GrLivArea"] > 4000) & (train_df["SalePrice"] < 300000))
].reset_index(drop=True)
train_df = train_df.drop(
    train_df[(train_df["GrLivArea"] > 4500) & (train_df["SalePrice"] < 500000)].index
).reset_index(drop=True)

y_train = np.log1p(train_df["SalePrice"])
train_df = train_df.drop(columns=["SalePrice"])

ntrain = train_df.shape[0]
all_df = pd.concat([train_df, test_df]).reset_index(drop=True)

MISSING VALUE IMPUTATION



In [3]:
cat_none_cols = [
    "Alley", "PoolQC", "Fence", "MiscFeature", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "MasVnrType"
]
for col in cat_none_cols:
    if col in all_df.columns:
        all_df[col] = all_df[col].fillna("None")

num_zero_cols = [
    "GarageYrBlt", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath",
    "GarageCars", "GarageArea"
]
for col in num_zero_cols:
    if col in all_df.columns:
        all_df[col] = all_df[col].fillna(0)

all_df["LotFrontage"] = all_df["LotFrontage"].fillna(all_df["LotFrontage"].median())
for col in all_df.select_dtypes(include=["object"]).columns:
    all_df[col] = all_df[col].fillna("None")
for col in all_df.select_dtypes(include=["number"]).columns:
    all_df[col] = all_df[col].fillna(0)

FEATURE ENGINEERING


In [4]:
all_df["TotalSF"] = all_df["TotalBsmtSF"] + all_df["1stFlrSF"] + all_df["2ndFlrSF"]
all_df["TotalBath"] = (
    all_df["BsmtFullBath"]
    + (0.5 * all_df["BsmtHalfBath"])
    + all_df["FullBath"]
    + (0.5 * all_df["HalfBath"])
)
all_df["HouseAge"] = all_df["YrSold"] - all_df["YearBuilt"]
all_df["RemodAge"] = all_df["YrSold"] - all_df["YearRemodAdd"]
all_df["TotalPorchSF"] = (
    all_df["OpenPorchSF"]
    + all_df["EnclosedPorch"]
    + all_df["3SsnPorch"]
    + all_df["ScreenPorch"]
)

all_df["Qual_TotalSF"] = all_df["OverallQual"] * all_df["TotalSF"]
all_df["Qual_GrLivArea"] = all_df["OverallQual"] * all_df["GrLivArea"]
all_df["Qual_GarageArea"] = all_df["OverallQual"] * all_df["GarageArea"]
all_df["Age_OverallQual"] = all_df["HouseAge"] * all_df["OverallQual"]

if "Id" in all_df.columns:
    all_df = all_df.drop(columns=["Id"])

SKEW CORRECTION AND ONE HOT ENCODING


In [5]:
numeric_feats = all_df.select_dtypes(include=[np.number]).columns
skewed_feats = all_df[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75].index
for feat in skewed_feats:
    if (all_df[feat] >= 0).all():
        all_df[feat] = np.log1p(all_df[feat])

all_df = pd.get_dummies(all_df).astype(float)
all_df = all_df.fillna(0)

X_train = all_df.iloc[:ntrain, :]
X_test = all_df.iloc[ntrain:, :]
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

TRAINING LOOP

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_preds_test = np.zeros(X_test.shape[0])
lgb_preds_test = np.zeros(X_test.shape[0])
lasso_preds_test = np.zeros(X_test.shape[0])
ridge_preds_test = np.zeros(X_test.shape[0])
enet_preds_test = np.zeros(X_test.shape[0])

blend_val_preds = np.zeros(ntrain)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_xgb = xgb.XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse",
        n_estimators=2500, learning_rate=0.012, max_depth=4,
        subsample=0.7, colsample_bytree=0.7, reg_alpha=0.6,
        reg_lambda=0.9, random_state=42, tree_method="hist"
    )
    model_xgb.fit(X_tr, y_tr)
    xgb_val = model_xgb.predict(X_va)
    xgb_preds_test += model_xgb.predict(X_test) / kf.n_splits

    model_lgb = lgb.LGBMRegressor(
        objective="regression", n_estimators=2000, learning_rate=0.015,
        max_depth=4, num_leaves=15, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.4, reg_lambda=0.6, random_state=42, verbose=-1
    )
    model_lgb.fit(X_tr, y_tr)
    lgb_val = model_lgb.predict(X_va)
    lgb_preds_test += model_lgb.predict(X_test) / kf.n_splits

    model_lasso = make_pipeline(
        RobustScaler(),
        LassoCV(alphas=[P * 1e-4 for P in range(1, 15)], random_state=42, cv=5, max_iter=5000)
    )
    model_lasso.fit(X_tr, y_tr)
    lasso_val = model_lasso.predict(X_va)
    lasso_preds_test += model_lasso.predict(X_test) / kf.n_splits

    model_ridge = make_pipeline(
        RobustScaler(),
        RidgeCV(alphas=[1, 5, 10, 20, 30, 50, 100], cv=5)
    )
    model_ridge.fit(X_tr, y_tr)
    ridge_val = model_ridge.predict(X_va)
    ridge_preds_test += model_ridge.predict(X_test) / kf.n_splits

    model_enet = make_pipeline(
        RobustScaler(),
        ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], alphas=[P * 1e-4 for P in range(1, 10)], cv=5, random_state=42, max_iter=5000)
    )
    model_enet.fit(X_tr, y_tr)
    enet_val = model_enet.predict(X_va)
    enet_preds_test += model_enet.predict(X_test) / kf.n_splits

    fold_blend_val = (
        (0.30 * xgb_val) +
        (0.25 * lgb_val) +
        (0.15 * lasso_val) +
        (0.15 * ridge_val) +
        (0.15 * enet_val)
    )
    blend_val_preds[val_idx] = fold_blend_val

overall_cv_rmse = np.sqrt(mean_squared_error(y_train, blend_val_preds))
print(f"Overall Out-of-Fold Blend CV RMSE: {overall_cv_rmse:.5f}")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.026528988864124692, tolerance: 0.015016870399739708
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.015536075660072868, tolerance: 0.014236502628131505
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.01795717540404773, tolerance: 0.014451081789759866
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:68

Overall Out-of-Fold Blend CV RMSE: 0.11036


In [7]:
final_blend_test = (
    (0.30 * xgb_preds_test) +
    (0.25 * lgb_preds_test) +
    (0.15 * lasso_preds_test) +
    (0.15 * ridge_preds_test) +
    (0.15 * enet_preds_test)
)

final_predictions = np.expm1(final_blend_test)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_predictions})
submission.to_csv("submission.csv", index=False)